In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!unzip "/content/drive/MyDrive/Dataset/kurdish_sign_dataset.zip" -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1387.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1388.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1389.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1390.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1391.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1392.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1393.JPG  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1394.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1395.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1396.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1397.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1398.jpg  
  inflating: /content/kurdish_sign_dataset/class_6/class_6_1399.jpg  
  inflating: /content/kurdish_sign_data

In [3]:
!ls /content

drive  kurdish_sign_dataset  sample_data


In [4]:
!ls /content/kurdish_sign_dataset/

class_1   class_14  class_19  class_23	class_28  class_32  class_6
class_10  class_15  class_2   class_24	class_29  class_33  class_7
class_11  class_16  class_20  class_25	class_3   class_34  class_8
class_12  class_17  class_21  class_26	class_30  class_4   class_9
class_13  class_18  class_22  class_27	class_31  class_5   class_mapping.txt


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [7]:
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [8]:
dataset = datasets.ImageFolder(
    root="/content/kurdish_sign_dataset",
    transform=transform
)

print("Number of classes:", len(dataset.classes))
print("Classes:", dataset.classes)

Number of classes: 34
Classes: ['class_1', 'class_10', 'class_11', 'class_12', 'class_13', 'class_14', 'class_15', 'class_16', 'class_17', 'class_18', 'class_19', 'class_2', 'class_20', 'class_21', 'class_22', 'class_23', 'class_24', 'class_25', 'class_26', 'class_27', 'class_28', 'class_29', 'class_3', 'class_30', 'class_31', 'class_32', 'class_33', 'class_34', 'class_4', 'class_5', 'class_6', 'class_7', 'class_8', 'class_9']


In [9]:
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [13]:
class CustomCNN(nn.Module):

    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),

            nn.Linear(128,256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256,num_classes)
        )

    def forward(self,x):

        x = self.features(x)
        x = self.classifier(x)

        return x

In [14]:
model = CustomCNN(34).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
epochs = 20

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")

Epoch 1/20, Loss: 0.4253414250873215
Epoch 2/20, Loss: 0.3916818173370551
Epoch 3/20, Loss: 0.3682290810257641
Epoch 4/20, Loss: 0.34275268915280777
Epoch 5/20, Loss: 0.3201717613496591
Epoch 6/20, Loss: 0.31439547584721106
Epoch 7/20, Loss: 0.3023774344826219
Epoch 8/20, Loss: 0.2843790111776015
Epoch 9/20, Loss: 0.2553342400943462
Epoch 10/20, Loss: 0.2497923413764185
Epoch 11/20, Loss: 0.26249920352922745
Epoch 12/20, Loss: 0.22688553326980984
Epoch 13/20, Loss: 0.22490029355409133
Epoch 14/20, Loss: 0.21023964435232811
Epoch 15/20, Loss: 0.19230854343344916
Epoch 16/20, Loss: 0.1872718482431191
Epoch 17/20, Loss: 0.19261343610190337
Epoch 18/20, Loss: 0.17117465012405642
Epoch 19/20, Loss: 0.17253015520634934
Epoch 20/20, Loss: 0.1769730979901048


In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100 * correct / total)

Validation Accuracy: 96.62464985994397


In [19]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

train_acc = 100 * correct / total

print("Training Accuracy:", train_acc)

Training Accuracy: 97.0199190787426


In [20]:
torch.save(model.state_dict(), "kusl_custom_cnn.pth")

In [21]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Dataset/kusl_custom_cnn.pth")